# S1 — Emergency NLP Service (API Specification & Multi-Epoch Training)
## نظام تحليل نصوص الطوارئ وفق مواصفات الـ REST API والمشروع المعتمدة

**1. مواصفات الخدمة والـ API Contract:**
- **Endpoint:** `POST /nlp/analyze`
- **Request:**
```json
{
  "emergency_id": "EMG-000001",
  "text": "يوجد حريق كبير في المنزل"
}
```
- **Response:**
```json
{
  "emergency_id": "EMG-000001",
  "text_urgency": 85,
  "emergency_type": "Fire",
  "priority": "critical",
  "entities": []
}
```

**2. الفئات المعتمدة لنوع الطوارئ (Emergency Types):**
- `Fire`
- `Flood`
- `Earthquake`
- `Traffic_Accident`
- `Medical_Emergency`
- `Other`
- `Normal`

**3. تدريب النموذج بعدة Epochs محلياً (100% Offline PyTorch).**

In [2]:
# ==============================================================================
# 1. المكتبات وإعداد الجهاز (Device)
# ==============================================================================
import os
import re
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch Version:", torch.__version__)
print("Device:", device)

PyTorch Version: 2.0.1+cpu
Device: cpu


In [3]:
# ==============================================================================
# 2. قراءة البيانات ومواءمتها مع الفئات الرسمية للمشروع
# ==============================================================================
df = pd.read_csv("HumAID_priority_training_ready_planB.csv")

# الفئات المعتمدة رسمياً في المشروع
OFFICIAL_EMERGENCY_TYPES = [
    "Fire",
    "Flood",
    "Earthquake",
    "Traffic_Accident",
    "Medical_Emergency",
    "Other",
    "Normal"
]
type_to_id = {t: idx for idx, t in enumerate(OFFICIAL_EMERGENCY_TYPES)}
id_to_type = {idx: t for idx, t in enumerate(OFFICIAL_EMERGENCY_TYPES)}

OFFICIAL_PRIORITIES = ["low", "medium", "high", "critical"]
prio_to_id = {p: idx for idx, p in enumerate(OFFICIAL_PRIORITIES)}
id_to_prio = {idx: p for idx, p in enumerate(OFFICIAL_PRIORITIES)}

def map_to_official_type(row):
    event = str(row.get("event", "")).lower()
    clabel = str(row.get("class_label", "")).lower()
    text = str(row.get("tweet_text", "")).lower()
    
    if clabel == "not_humanitarian":
        return "Normal"
    if any(k in text for k in ["traffic", "crash", "collision", "car accident", "highway accident", "overturned vehicle"]):
        return "Traffic_Accident"
    if any(k in text for k in ["injured", "dead", "casualties", "ambulance", "bleeding", "hospital patient", "medical help"]):
        return "Medical_Emergency"
    if "wildfire" in event or any(k in text for k in ["fire", "wildfire", "blaze", "burning", "flames", "smoke"]):
        return "Fire"
    if "earthquake" in event or any(k in text for k in ["earthquake", "quake", "tremor", "aftershock", "seismic", "rubble"]):
        return "Earthquake"
    if any(e in event for e in ["flood", "hurricane", "cyclone"]) or any(k in text for k in ["flood", "flooding", "water rising", "drowning", "storm"]):
        return "Flood"
    return "Other"

def map_to_official_priority(row):
    clabel = str(row.get("class_label", "")).lower()
    text = str(row.get("tweet_text", "")).lower()
    
    if clabel == "injured_or_dead_people" or any(k in text for k in ["dead", "killed", "casualties", "fatalities", "trapped", "dying", "bleeding"]):
        return "critical"
    if clabel in ["requests_or_urgent_needs", "infrastructure_and_utility_damage", "displaced_people_and_evacuations"]:
        return "high"
    if clabel in ["caution_and_advice", "rescue_volunteering_or_donation_effort", "other_relevant_information"]:
        return "medium"
    return "low"

df["mapped_type"] = df.apply(map_to_official_type, axis=1)
df["type_id"] = df["mapped_type"].map(type_to_id)

df["mapped_priority"] = df.apply(map_to_official_priority, axis=1)
df["priority_id"] = df["mapped_priority"].map(prio_to_id)

train_df = df[df["split"] == "train"].copy().reset_index(drop=True)
dev_df = df[df["split"] == "dev"].copy().reset_index(drop=True)
test_df = df[df["split"] == "test"].copy().reset_index(drop=True)

print(f"Dataset: {len(df)} rows | Train: {len(train_df)} | Dev: {len(dev_df)} | Test: {len(test_df)}")
print("Emergency Types in Train:", train_df["mapped_type"].value_counts().to_dict())
print("Priority Levels in Train:", train_df["mapped_priority"].value_counts().to_dict())

Dataset: 76484 rows | Train: 53531 | Dev: 7793 | Test: 15160
Emergency Types in Train: {'Flood': 35233, 'Fire': 5829, 'Earthquake': 5649, 'Normal': 4407, 'Medical_Emergency': 2332, 'Traffic_Accident': 81}
Priority Levels in Train: {'medium': 26690, 'low': 10506, 'high': 10152, 'critical': 6183}


In [4]:
# ==============================================================================
# 3. بناء المعجم النصي والترميز (Vocabulary & Tokenization)
# ==============================================================================
VOCAB_SIZE = 6000
MAX_LEN = 50
PAD_IDX = 0
UNK_IDX = 1

word_counts = {}
for text in train_df["tweet_text"].fillna(""):
    for word in str(text).lower().strip().split():
        word_counts[word] = word_counts.get(word, 0) + 1

sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)[:VOCAB_SIZE - 2]
token_to_id = {word: idx + 2 for idx, (word, _) in enumerate(sorted_words)}
token_to_id["<PAD>"] = PAD_IDX
token_to_id["<UNK>"] = UNK_IDX

def text_to_sequence(text, max_len=MAX_LEN):
    tokens = str(text).lower().strip().split()
    seq = [token_to_id.get(w, UNK_IDX) for w in tokens[:max_len]]
    if len(seq) < max_len:
        seq += [PAD_IDX] * (max_len - len(seq))
    return seq

class EmergencyDataset(Dataset):
    def __init__(self, texts, types, priorities):
        self.seqs = [text_to_sequence(t) for t in texts]
        self.types = torch.tensor(types, dtype=torch.long)
        self.prios = torch.tensor(priorities, dtype=torch.long)
        
    def __len__(self):
        return len(self.seqs)
        
    def __getitem__(self, idx):
        return torch.tensor(self.seqs[idx], dtype=torch.long), self.types[idx], self.prios[idx]

train_loader = DataLoader(EmergencyDataset(train_df["tweet_text"], train_df["type_id"], train_df["priority_id"]), batch_size=64, shuffle=True)
dev_loader = DataLoader(EmergencyDataset(dev_df["tweet_text"], dev_df["type_id"], dev_df["priority_id"]), batch_size=64, shuffle=False)
test_loader = DataLoader(EmergencyDataset(test_df["tweet_text"], test_df["type_id"], test_df["priority_id"]), batch_size=64, shuffle=False)

print(f"Vocabulary built with {len(token_to_id)} tokens. DataLoaders ready.")

Vocabulary built with 6000 tokens. DataLoaders ready.


In [5]:
# ==============================================================================
# 4. معمارية نموذج BiLSTM + Masked Attention متعدد المهام
# ==============================================================================
class MaskedAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Linear(hidden_dim * 2, 1)

    def forward(self, lstm_output, mask):
        attn_scores = self.attention(lstm_output).squeeze(-1)
        masked_scores = attn_scores.masked_fill(~mask, -1e9)
        attn_weights = F.softmax(masked_scores, dim=1).unsqueeze(-1)
        context = torch.sum(attn_weights * lstm_output, dim=1)
        return context, attn_weights

class S1EmergencyNLPModel(nn.Module):
    def __init__(self, vocab_size=6000, embedding_dim=100, hidden_dim=128, num_types=7, num_priorities=4, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=PAD_IDX)
        self.bilstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        self.attention = MaskedAttention(hidden_dim)
        self.dropout = nn.Dropout(dropout)
        
        # Heads: Emergency Type & Priority
        self.type_head = nn.Linear(hidden_dim * 2, num_types)
        self.priority_head = nn.Linear(hidden_dim * 2, num_priorities)

    def forward(self, x):
        mask = (x != PAD_IDX)
        embedded = self.dropout(self.embedding(x))
        lstm_out, _ = self.bilstm(embedded)
        text_embedding, attn_weights = self.attention(lstm_out, mask)
        dropped = self.dropout(text_embedding)
        
        type_logits = self.type_head(dropped)
        priority_logits = self.priority_head(dropped)
        return type_logits, priority_logits, text_embedding

model = S1EmergencyNLPModel(
    vocab_size=VOCAB_SIZE,
    embedding_dim=100,
    hidden_dim=128,
    num_types=len(OFFICIAL_EMERGENCY_TYPES),
    num_priorities=len(OFFICIAL_PRIORITIES)
).to(device)

print("Model Architecture Initialized.")

Model Architecture Initialized.


In [6]:
# ==============================================================================
# 5. دورة التدريب مع حساب أوزان الفئات الآمنة (Training Loop with Safe Weights)
# ==============================================================================
# حساب الأوزان بشكل آمن لتفادي أخطاء الفئات غير الموجودة
present_type_classes = np.unique(train_df["type_id"].values)
w_type = compute_class_weight("balanced", classes=present_type_classes, y=train_df["type_id"].values)
type_weights = np.ones(len(OFFICIAL_EMERGENCY_TYPES), dtype=np.float32)
type_weights[present_type_classes] = w_type

present_prio_classes = np.unique(train_df["priority_id"].values)
w_prio = compute_class_weight("balanced", classes=present_prio_classes, y=train_df["priority_id"].values)
prio_weights = np.ones(len(OFFICIAL_PRIORITIES), dtype=np.float32)
prio_weights[present_prio_classes] = w_prio

criterion_type = nn.CrossEntropyLoss(weight=torch.tensor(type_weights, dtype=torch.float32).to(device))
criterion_prio = nn.CrossEntropyLoss(weight=torch.tensor(prio_weights, dtype=torch.float32).to(device))
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS = 15
best_val_loss = float("inf")
print("=" * 70)
print(f"Starting Model Training for {EPOCHS} Epochs on Device: {device}")
print("=" * 70)

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    
    for batch_x, batch_type, batch_prio in train_loader:
        batch_x = batch_x.to(device)
        batch_type = batch_type.to(device)
        batch_prio = batch_prio.to(device)
        
        optimizer.zero_grad()
        type_logits, prio_logits, _ = model(batch_x)
        
        loss = criterion_type(type_logits, batch_type) + criterion_prio(prio_logits, batch_prio)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    avg_train_loss = train_loss / len(train_loader)
    
    # Validation Loop
    model.eval()
    val_loss = 0.0
    val_type_correct = 0
    val_prio_correct = 0
    total_val = 0
    
    with torch.no_grad():
        for batch_x, batch_type, batch_prio in dev_loader:
            batch_x = batch_x.to(device)
            batch_type = batch_type.to(device)
            batch_prio = batch_prio.to(device)
            
            type_logits, prio_logits, _ = model(batch_x)
            loss = criterion_type(type_logits, batch_type) + criterion_prio(prio_logits, batch_prio)
            val_loss += loss.item()
            
            val_type_correct += (type_logits.argmax(dim=-1) == batch_type).sum().item()
            val_prio_correct += (prio_logits.argmax(dim=-1) == batch_prio).sum().item()
            total_val += batch_x.size(0)
            
    avg_val_loss = val_loss / len(dev_loader)
    val_type_acc = val_type_correct / total_val
    val_prio_acc = val_prio_correct / total_val
    
    print(f"Epoch [{epoch:02d}/{EPOCHS:02d}] | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Type Acc: {val_type_acc*100:.2f}% | Val Priority Acc: {val_prio_acc*100:.2f}%")
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save({
            "model_state_dict": model.state_dict(),
            "token_to_id": token_to_id,
            "official_types": OFFICIAL_EMERGENCY_TYPES,
            "official_priorities": OFFICIAL_PRIORITIES,
            "epoch": epoch,
            "val_loss": best_val_loss
        }, "s1_emergency_bilstm_model0.2.pt")

print("=" * 70)
print("Training Complete. Best Model Checkpoint Saved to 's1_emergency_bilstm_model.pt'.")

Starting Model Training for 15 Epochs on Device: cpu
Epoch [01/15] | Train Loss: 1.9623 | Val Loss: 1.4531 | Val Type Acc: 63.43% | Val Priority Acc: 69.38%
Epoch [02/15] | Train Loss: 1.3058 | Val Loss: 1.2721 | Val Type Acc: 67.66% | Val Priority Acc: 75.38%
Epoch [03/15] | Train Loss: 1.1275 | Val Loss: 1.1372 | Val Type Acc: 78.84% | Val Priority Acc: 78.54%
Epoch [04/15] | Train Loss: 1.0075 | Val Loss: 1.1199 | Val Type Acc: 73.63% | Val Priority Acc: 77.09%
Epoch [05/15] | Train Loss: 0.9135 | Val Loss: 1.0912 | Val Type Acc: 78.87% | Val Priority Acc: 79.51%
Epoch [06/15] | Train Loss: 0.8602 | Val Loss: 1.1245 | Val Type Acc: 80.57% | Val Priority Acc: 79.61%
Epoch [07/15] | Train Loss: 0.8086 | Val Loss: 1.0705 | Val Type Acc: 81.69% | Val Priority Acc: 79.08%
Epoch [08/15] | Train Loss: 0.7691 | Val Loss: 1.0869 | Val Type Acc: 80.82% | Val Priority Acc: 78.98%
Epoch [09/15] | Train Loss: 0.7250 | Val Loss: 1.1134 | Val Type Acc: 80.02% | Val Priority Acc: 79.47%
Epoch [10/1

In [7]:
# ==============================================================================
# 6. التقييم على مجموعة الاختبار الرسمية (Test Set Evaluation)
# ==============================================================================
checkpoint = torch.load("s1_emergency_bilstm_model.pt", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

test_type_true, test_type_pred = [], []
test_prio_true, test_prio_pred = [], []

with torch.no_grad():
    for batch_x, batch_type, batch_prio in test_loader:
        batch_x = batch_x.to(device)
        type_logits, prio_logits, _ = model(batch_x)
        
        test_type_true.extend(batch_type.numpy())
        test_type_pred.extend(type_logits.argmax(dim=-1).cpu().numpy())
        
        test_prio_true.extend(batch_prio.numpy())
        test_prio_pred.extend(prio_logits.argmax(dim=-1).cpu().numpy())

acc_type = accuracy_score(test_type_true, test_type_pred)
acc_prio = accuracy_score(test_prio_true, test_prio_pred)

print("=" * 75)
print(f"Official Test Set Evaluation ({len(test_df)} samples)")
print(f"Emergency Type Classification Accuracy : {acc_type * 100:.2f}%")
print(f"Priority Level Classification Accuracy  : {acc_prio * 100:.2f}%")
print("=" * 75)

print("\nClassification Report: Emergency Types:")
# print(classification_report(test_type_true, test_type_pred, target_names=OFFICIAL_EMERGENCY_TYPES, zero_division=0))

print("Classification Report: Priority Levels:")
print(classification_report(test_prio_true, test_prio_pred, target_names=OFFICIAL_PRIORITIES, zero_division=0))

Official Test Set Evaluation (15160 samples)
Emergency Type Classification Accuracy : 73.86%
Priority Level Classification Accuracy  : 74.75%

Classification Report: Emergency Types:
Classification Report: Priority Levels:
              precision    recall  f1-score   support

         low       0.61      0.68      0.64      2966
      medium       0.81      0.76      0.78      7567
        high       0.66      0.69      0.68      2875
    critical       0.91      0.91      0.91      1752

    accuracy                           0.75     15160
   macro avg       0.75      0.76      0.75     15160
weighted avg       0.75      0.75      0.75     15160



In [8]:
# ==============================================================================
# 7. دوال استخراج الكيانات وحساب نقاط الإلحاحية (Entities & Urgency Score)
# ==============================================================================
def extract_entities(text: str) -> list:
    """
    استخراج الكيانات المذكورة في النص (أرقام، أعداد مصابين، قطاعات ومواقع)
    """
    entities = []
    t = str(text).strip()
    
    # 1. استخراج الأعداد والمصابين/العائلات
    num_matches = re.findall(r'\b\d+\s*(?:people|persons|casualties|injured|dead|families|cars|vehicles|floors?|houses?)?\b', t, re.I)
    for m in num_matches:
        if m.strip() and m.strip() not in entities:
            entities.append(m.strip())
            
    # 2. استخراج أسماء المواقع أو القطاعات
    loc_matches = re.findall(r'\b(?:sector|street|highway|block|district|city|hospital|bridge|area|road)\s*[\w\d-]+\b', t, re.I)
    for loc in loc_matches:
        if loc.strip() and loc.strip() not in entities:
            entities.append(loc.strip())
            
    return entities

def compute_text_urgency_score(priority_probabilities: dict) -> int:
    """
    حساب درجة إلحاحية النص كعدد صحيح من 0 إلى 100
    """
    score = (
        priority_probabilities.get("critical", 0.0) * 100 +
        priority_probabilities.get("high", 0.0) * 75 +
        priority_probabilities.get("medium", 0.0) * 45 +
        priority_probabilities.get("low", 0.0) * 10
    )
    return int(np.clip(round(score), 0, 100))

print("Entity Extractor and Urgency Score functions ready.")

Entity Extractor and Urgency Score functions ready.


In [9]:
# ==============================================================================
# 8. دالة الـ NLP API الرئيسية المطابقة للمواصفات الرسمية للمشروع
#    POST /nlp/analyze
# ==============================================================================
@torch.inference_mode()
def analyze_emergency(request_json: dict) -> dict:
    """
    تنفيذ معالجة الطلب بالكامل ومطابقة مواصفات S1 NLP Response:
    Input:  {"emergency_id": "EMG-000001", "text": "..."}
    Output: {"emergency_id": "...", "text_urgency": 85, "emergency_type": "Fire", "priority": "critical", "entities": []}
    """
    emergency_id = request_json.get("emergency_id", "EMG-000001")
    text = request_json.get("text", "")
    
    seq = text_to_sequence(text, max_len=MAX_LEN)
    seq_tensor = torch.tensor([seq], dtype=torch.long, device=device)
    
    type_logits, priority_logits, _ = model(seq_tensor)
    
    # 1. استنتاج نوع الطوارئ من الفئات الرسمية
    type_probs = F.softmax(type_logits, dim=-1)[0].cpu().numpy()
    detected_type = id_to_type[int(np.argmax(type_probs))]
    
    # 2. حساب الاحتماليات واختيار الأولوية ذات النسبة الأعلى
    prio_probs = F.softmax(priority_logits, dim=-1)[0].cpu().numpy()
    priority_probabilities = {
        id_to_prio[i]: round(float(prob), 4) for i, prob in enumerate(prio_probs)
    }
    
    # اختيار الفئة ذات النسبة الأعلى
    top_priority = max(priority_probabilities, key=priority_probabilities.get)
    
    # 3. حساب درجة الـ text_urgency (0-100)
    text_urgency = compute_text_urgency_score(priority_probabilities)
    
    # 4. استخراج الكيانات المذكورة
    entities = extract_entities(text)
    
    # 5. تكوين الاستجابة الرسمية للـ API
    return {
        "emergency_id": emergency_id,
        "text_urgency": text_urgency,
        "emergency_type": detected_type,
        "priority": top_priority,
        "entities": entities
    }

print("analyze_emergency (POST /nlp/analyze) ready.")

analyze_emergency (POST /nlp/analyze) ready.


In [10]:
# ==============================================================================
# 9. اختبار استجابة الـ API على سيناريوهات رسمية مطابقة لشكل الطلب
# ==============================================================================
api_requests = [
    {
        "emergency_id": "EMG-000001",
        "text": "Major fire broke out in the residential building, smoke everywhere and families need rescue."
    },
    {
        "emergency_id": "EMG-000002",
        "text": "Flash flood water rising rapidly in sector 4, 15 people trapped on the roof without food."
    },
    {
        "emergency_id": "EMG-000003",
        "text": "Multiple car collision on highway 60 with severe casualties and road completely blocked."
    },
    {
        "emergency_id": "EMG-000004",
        "text": "Building collapsed after powerful earthquake tremor, multiple people injured under rubble."
    },
    {
        "emergency_id": "EMG-000005",
        "text": "What is the weather forecast for tomorrow morning in the city?"
    }
]

print("=" * 75)
print("REST API ENDPOINT RESPONSE (POST /nlp/analyze)")
print("=" * 75)

for req in api_requests:
    resp = analyze_emergency(req)
    print(f"\n[REQUEST] : ID={req['emergency_id']} | Text: {req['text']}")
    print("[RESPONSE]:")
    print(json.dumps(resp, indent=2, ensure_ascii=False))
    print("-" * 75)

REST API ENDPOINT RESPONSE (POST /nlp/analyze)

[REQUEST] : ID=EMG-000001 | Text: Major fire broke out in the residential building, smoke everywhere and families need rescue.
[RESPONSE]:
{
  "emergency_id": "EMG-000001",
  "text_urgency": 51,
  "emergency_type": "Fire",
  "priority": "low",
  "entities": []
}
---------------------------------------------------------------------------

[REQUEST] : ID=EMG-000002 | Text: Flash flood water rising rapidly in sector 4, 15 people trapped on the roof without food.
[RESPONSE]:
{
  "emergency_id": "EMG-000002",
  "text_urgency": 100,
  "emergency_type": "Flood",
  "priority": "critical",
  "entities": [
    "4",
    "15 people",
    "sector 4"
  ]
}
---------------------------------------------------------------------------

[REQUEST] : ID=EMG-000003 | Text: Multiple car collision on highway 60 with severe casualties and road completely blocked.
[RESPONSE]:
{
  "emergency_id": "EMG-000003",
  "text_urgency": 35,
  "emergency_type": "Normal",
  "

In [11]:
# ==============================================================================
# 10. خلية تفاعلية لإدخال المستخدم واختبار استجابة الـ API مباشرة
# ==============================================================================
user_id = input("Enter Emergency ID (e.g. EMG-000001): ").strip() or "EMG-000001"
user_text = input("Enter Emergency Description Text: ").strip()

if user_text:
    req = {"emergency_id": user_id, "text": user_text}
    response = analyze_emergency(req)
    
    print("\n" + "=" * 50)
    print("NLP SERVICE RESPONSE (JSON):")
    print("=" * 50)
    print(json.dumps(response, indent=4, ensure_ascii=False))
    print("=" * 50)
else:
    print("No text provided.")


NLP SERVICE RESPONSE (JSON):
{
    "emergency_id": "E002",
    "text_urgency": 41,
    "emergency_type": "Earthquake",
    "priority": "low",
    "entities": []
}
